# 00 · Setup, Data Loading & Reproducible Split

**Course:** Data Mining (DA) — Autumn 2026  
**Group:** 1  
**Topic:** Credit Card Fraud Detection (Option E)  
**Dataset:** `creditcard.csv` — Kaggle (ULB), 284,807 transactions

**Mục tiêu notebook này (bước S0 + S1 + S2 của PIPELINE.md):**
1. Kiểm tra môi trường và cấu trúc thư mục
2. Load dữ liệu, kiểm tra chất lượng (missing / duplicates)
3. **Stratified split 70/30** — tách tập đúng cách, KHÔNG rò rỉ
4. Lưu split vào `artifacts/models/split.joblib` để mọi notebook sau dùng chung

> ⚠️ **Nguyên tắc quan trọng nhất của toàn bộ project:**
> `clean → stratified split → resample CHỈ trên train`.
> Resample trước rồi mới split sẽ làm mọi kết quả về sau vô giá trị.

In [1]:
import sys, warnings, platform, os
warnings.filterwarnings("ignore")

print(f"Python: {sys.version.split()[0]}  ({platform.system()} {platform.machine()})")
print("\n=== ENVIRONMENT CHECK ===")
missing = []
for m in ["pandas", "numpy", "sklearn", "imblearn", "xgboost", "shap",
          "matplotlib", "seaborn", "scipy", "joblib"]:
    try:
        mod = __import__(m)
        print(f"  {m:12s} {getattr(mod, '__version__', '?'):>10s}  OK")
    except ImportError:
        print(f"  {m:12s} {'MISSING':>10s}  <-- cai dat")
        missing.append(m)

if missing:
    print("\n[!] Chay lenh sau de cai:")
    print("    python -m pip install " + " ".join(missing))

Python: 3.11.9  (Windows AMD64)

=== ENVIRONMENT CHECK ===


  pandas            2.3.3  OK
  numpy             2.2.6  OK


  sklearn           1.9.1  OK


  imblearn         0.14.2  OK
  xgboost           3.2.0  OK


  shap             0.51.0  OK
  matplotlib       3.10.9  OK
  seaborn          0.13.2  OK
  scipy            1.17.1  OK
  joblib            1.6.0  OK


## 1. Import module dùng chung

Thêm thư mục gốc project vào `sys.path` để `from src.xxx import ...` hoạt động trong notebook.

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path(r"e:\Projects\DataMining")
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.config import (DATA_CSV, SEED, PALETTE, TABLES, MODELS, CHARTS,
                        COST_PER_FN, COST_PER_FP, MIN_PRECISION)
from src.data import (load_raw, clean, make_split, save_split, load_split,
                      print_split_summary, quality_report)
from src import features as feat

sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.figsize": (14, 6),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "font.family": "DejaVu Sans",
})

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_CSV     :", DATA_CSV, "exists =", DATA_CSV.exists())
print("SEED         :", SEED)
print("Cost/FN = %.2f USD | Cost/FP = %.2f USD" % (COST_PER_FN, COST_PER_FP))

PROJECT_ROOT : e:\Projects\DataMining
DATA_CSV     : E:\Projects\DataMining\creditcard.csv exists = True
SEED         : 42
Cost/FN = 5.45 USD | Cost/FP = 0.30 USD


## 2. Load raw data

Kiểm tra shape, dtypes, missing values và duplicates.

In [3]:
df_raw = load_raw()
print("Shape:", df_raw.shape)
print("Memory: %.2f MB" % (df_raw.memory_usage(deep=True).sum() / 1024**2))
print("\nDtypes:")
print(df_raw.dtypes.value_counts().to_string())
print("\nTarget distribution (RAW):")
print(df_raw['Class'].value_counts().to_string())
print("\nMissing values total:", int(df_raw.isna().sum().sum()))
print("Duplicate rows:", int(df_raw.duplicated().sum()))
df_raw.head(3)

Shape: (284807, 31)
Memory: 67.36 MB

Dtypes:
float64    30
int64       1

Target distribution (RAW):
Class
0    284315
1       492

Missing values total: 0


Duplicate rows: 1081


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0


## 3. Data quality report

Bảng QC này đưa thẳng vào report cuối kỳ (mục 6 — EDA Findings).

In [4]:
df_clean = clean(df_raw)

qc = quality_report(df_raw, df_clean)
display(qc)
qc.to_csv(TABLES / "data_quality_report.csv", index=False)
print("[saved]", TABLES / "data_quality_report.csv")

[clean] removed 1081 duplicate rows -> (283726, 31)


,check,value
0,shape_raw,"(284807, 31)"
1,shape_clean,"(283726, 31)"
2,missing_values_total,0
3,duplicate_rows,1081
4,duplicates_in_legit,1062
5,duplicates_in_fraud,19
6,fraud_count_clean,473
7,legit_count_clean,283253
8,imbalance_ratio_1_to,598.8
9,amount_negative,0


[saved] E:\Projects\DataMining\artifacts\tables\data_quality_report.csv


### Vì sao drop duplicates? (câu hỏi thầy chắc chắn sẽ hỏi)

* 1.081 dòng trùng: **1.062 ở class 0** (0.37%) và **19 ở class 1** (3.66%).
* Giữ lại → mô hình **overfit** các mẫu lặp và **lệch độ nhạy** (recall) của lớp fraud.
* Dùng `stratify=y` nên **tỷ lệ class không thay đổi** sau khi tách tập.
* Sau clean: **283,726** dòng, imbalance **1 : 599**.

> Lưu ý số liệu: slide Progress Report 1 dùng **1:577 / 492 fraud** (số RAW),
> còn notebook EDA dùng **1:599 / 473 fraud** (số SAU CLEAN). Cần chốt 1 nguồn
> cho report cuối kỳ — khuyến nghị dùng **số sau clean** và ghi rõ.

In [5]:
dup = df_raw.duplicated()
print("Duplicates by class:")
print(df_raw[dup].groupby('Class').size().to_string())
print("\nClass balance AFTER cleaning:")
cc = df_clean['Class'].value_counts().sort_index()
for c in [0, 1]:
    print(f"  Class {c}: {cc[c]:>8,}  ({cc[c]/len(df_clean)*100:.4f}%)")
print(f"  Imbalance ratio = 1 : {cc[0]/cc[1]:.1f}")

Duplicates by class:
Class
0    1062
1      19

Class balance AFTER cleaning:
  Class 0:  283,253  (99.8333%)
  Class 1:      473  (0.1667%)
  Imbalance ratio = 1 : 598.8


## 4. Kiểm tra rò rỉ dữ liệu (data leakage)

Câu hỏi này luôn xuất hiện trong phần Q&A. Kiểm tra trước khi đi tiếp:

In [6]:
leak_checks = pd.DataFrame({
    "check": [
        "target 'Class' in features",
        "feature count (expect 30)",
        "negative Amount rows",
        "NaN in Time",
        "NaN in Amount",
        "Time monotonic increasing",
        "Time range (seconds)",
    ],
    "result": [
        "Class" in df_clean.drop(columns=['Class']).columns,
        df_clean.shape[1] - 1,
        int((df_clean['Amount'] < 0).sum()),
        int(df_clean['Time'].isna().sum()),
        int(df_clean['Amount'].isna().sum()),
        bool(df_clean['Time'].is_monotonic_increasing),
        f"{df_clean['Time'].min():.0f} -> {df_clean['Time'].max():.0f} (~2 days)",
    ],
})
display(leak_checks)

print("Ghi chu:")
print("- 'Time' la giay ke tu giao dich dau -> KHONG phai timestamp that,")
print("  nen khong the suy ra ngay giao dich ngoai 2 ngay duoc viet.")
print("- Khong co feature nao bi 'leak' truc tiep tu outcome.")
print("- Scaler se nam TRONG Pipeline (buoc S4) -> fit tren train, khong ro ri.")

,check,result
0,target 'Class' in features,False
1,feature count (expect 30),30
2,negative Amount rows,0
3,NaN in Time,0
4,NaN in Amount,0
5,Time monotonic increasing,True
6,Time range (seconds),0 -> 172792 (~2 days)


Ghi chu:
- 'Time' la giay ke tu giao dich dau -> KHONG phai timestamp that,
  nen khong the suy ra ngay giao dich ngoai 2 ngay duoc viet.
- Khong co feature nao bi 'leak' truc tiep tu outcome.
- Scaler se nam TRONG Pipeline (buoc S4) -> fit tren train, khong ro ri.


## 5. ⭐ STRATIFIED SPLIT — bước quan trọng nhất

**Tại sao stratify?** Với imbalance 1:599, một tập 30% không stratify có thể chỉ còn
vài chục fraud → metric không ổn định, độ ngẫu nhiên cao.

**Tại sao random split chứ không time split?** Dal Pozzolo et al. (2014) và
Leal-Avila et al. (2023) dùng đúng protocol stratified random split trên dataset
này → kết quả của ta so sánh được với literature. Time-split sẽ chạy thêm như
một robustness check ở bước S7.

In [7]:
X_train, X_test, y_train, y_test = make_split(df_clean, test_size=0.30, seed=SEED)

print_split_summary(y_train, y_test)

print("\nKy vong (doi chieu):")
print("  train: n=198,608  fraud=331  rate=0.1666%")
print("  test : n=85,118   fraud=142  rate=0.1669%")

train: n=198,608  fraud= 331  rate=0.1667%  imbalance=1:599


 test: n= 85,118  fraud= 142  rate=0.1668%  imbalance=1:598

Ky vong (doi chieu):
  train: n=198,608  fraud=331  rate=0.1666%
  test : n=85,118   fraud=142  rate=0.1669%


## 6. Feature engineering (S3)

5 feature mới, mỗi cái đều xuất phát từ một phát hiện cụ thể trong EDA:

| Feature | Công thức | Lý do |
|---|---|---|
| `Hour_of_Day` | `(Time/3600).astype(int) % 24` | Peak fraud rate ở **giờ 2** (1.45%) |
| `Day` | `(Time/86400).astype(int)` | Day 0: 272 fraud vs Day 1: 201 |
| `Is_Night` | `Hour ∈ {22..5}` | Nhóm giờ thấp điểm gian lận |
| `Amount_log` | `log1p(Amount)` | Amount right-skewed (max 25,691 vs median 22) |
| `Is_Zero_Amount` | `Amount == 0` | 1,695 giao dịch 0 USD là pattern bất thường |

> Không thêm `Amount_z` — StandardScaler trong Pipeline đã làm đúng việc đó.

In [8]:
X_train = feat.add_features(X_train)
X_test  = feat.add_features(X_test)

print("X_train:", X_train.shape, "  X_test:", X_test.shape)
print("New features:", feat.NEW_FEATURES)
X_train[feat.NEW_FEATURES].describe().round(3)

X_train: (198608, 35)   X_test: (85118, 35)
New features: ['Hour_of_Day', 'Day', 'Is_Night', 'Amount_log', 'Is_Zero_Amount']


,Hour_of_Day,Day,Is_Night,Amount_log,Is_Zero_Amount
count,198608.000,198608.000,198608.000,198608.000,198608.000
mean,14.051,0.493,0.176,3.156,0.006
std,5.829,0.500,0.381,1.656,0.080
min,0.000,0.000,0.000,0.000,0.000
25%,10.000,0.000,0.000,1.902,0.000
50%,15.000,0.000,0.000,3.139,0.000
75%,19.000,1.000,0.000,4.366,0.000
max,23.000,1.000,1.000,9.847,1.000


In [9]:
mi = feat.mutual_information_table(X_train, y_train, top_n=20)
mi.to_csv(TABLES / "mutual_information.csv")
print("Top 20 features theo mutual information voi target:\n")
print(mi.round(5).to_string())
print("\n[saved]", TABLES / "mutual_information.csv")

print("\nVi tri cua feature moi (nen nam trong top 15-20):")
for f in feat.NEW_FEATURES:
    rank = (mi.index.tolist() + list(X_train.columns)).index(f) + 1
    print(f"  {f:16s} rank ~{rank}")

Top 20 features theo mutual information voi target:

Day            0.03247
V17            0.00802
V14            0.00798
V10            0.00744
V12            0.00737
V11            0.00657
V16            0.00591
V4             0.00475
V3             0.00468
V9             0.00406
V18            0.00401
Is_Night       0.00400
V7             0.00378
V2             0.00303
Hour_of_Day    0.00271
V27            0.00223
V21            0.00221
V6             0.00204
V1             0.00194
V5             0.00193

[saved] E:\Projects\DataMining\artifacts\tables\mutual_information.csv

Vi tri cua feature moi (nen nam trong top 15-20):
  Hour_of_Day      rank ~15
  Day              rank ~1
  Is_Night         rank ~12
  Amount_log       rank ~54
  Is_Zero_Amount   rank ~55


In [10]:
feat.feature_evidence_chart(X_train, y_train, CHARTS / "feature_engineering_evidence.png")

[features] chart saved -> E:\Projects\DataMining\artifacts\charts\feature_engineering_evidence.png


## 7. Lưu split + feature cho mọi notebook sau

**Quy tắc bất di bất dịch:** không notebook nào được tự tách tập riêng.
Tất cả import từ `artifacts/models/split.joblib`. Nếu không, kết quả ở các bước
S5–S9 sẽ không so sánh được với nhau.

In [11]:
save_split(X_train, X_test, y_train, y_test)

# Luu them ban khong feature de notebook sau dung dung feature set
import joblib
joblib.dump(
    {"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test,
     "feature_names": list(X_train.columns)},
    MODELS / "split_with_features.joblib"
)
print("[saved]", MODELS / "split_with_features.joblib")

print("\n=== VERIFY RELOAD ===")
d = load_split()
print("train:", d['X_train'].shape, "test:", d['X_test'].shape)
print("OK — split da sanh sang cho moi notebook.")

[split] saved -> E:\Projects\DataMining\artifacts\models\split.joblib
[saved] E:\Projects\DataMining\artifacts\models\split_with_features.joblib

=== VERIFY RELOAD ===
train: (198608, 35) test: (85118, 35)
OK — split da sanh sang cho moi notebook.


## 8. ✅ Definition of Done — bước S0/S1/S2/S3

- [x] Môi trường đủ package, cấu trúc thư mục đã tạo
- [x] Load + clean: 283,726 dòng, 0 missing, đã drop 1,081 duplicates
- [x] Bảng QC lưu `artifacts/tables/data_quality_report.csv`
- [x] Kiểm tra data leakage — không rò rỉ
- [x] **Stratified split 70/30**, tỷ lệ fraud khớp ở cả 2 tập
- [x] Split lưu `artifacts/models/split.joblib`
- [x] 5 feature mới + bảng mutual information + chart bằng chứng

### Bước tiếp theo

→ `01_preprocessing.ipynb`: dựng ColumnTransformer + StandardScaler, so sánh 3
biến thể scaler, kiểm tra biến thể feature (có/bỏ feature mới) trên tập train.